Fundamentals of Deep Learning Models

# Lab 02-3: Support Vector Machine
## Exercise: Predicting Iris Species

This exercise implements a **linear soft-margin SVM** (Section 2.7) for binary classification using the hinge loss (Eq. 2.36) and subgradient descent (Eqs. 2.38–2.39). The SVM learns a separating hyperplane $\mathbf{w}^T\mathbf{x} + b = 0$ that maximizes the margin between two classes.

### Prepare IRIS Dataset

For binary SVM classification, we use class labels $y \in \{+1, -1\}$ (Section 2.6). Setosa is labeled $-1$ and versicolor is labeled $+1$; virginica samples are removed.

In [1]:
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version

print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)
print('scikit-learn version:', sklearn_version)

NumPy version: 2.0.2
Pandas version: 2.2.2
scikit-learn version: 1.6.1


In [2]:
from sklearn.datasets import load_iris

iris = load_iris()

# iris.data columns: sepal length, sepal width, petal length, petal width (all in cm)
# iris.target: species label (0=setosa, 1=versicolor, 2=virginica)
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_tf = pd.DataFrame(data=iris.target, columns=['species'])

# Convert to SVM labels: versicolor=+1, setosa=-1 (Eq. 2.28)
def converter(species):
    return 1 if species == 1 else -1

iris_tf['species'] = iris_tf['species'].apply(converter)

# Remove virginica samples (indices 100-149) for binary classification
REMOVE_virginica = True

if REMOVE_virginica:
    iris_df = iris_df.drop(labels=range(100,150), axis=0)
    iris_tf = iris_tf.drop(labels=range(100,150), axis=0)

# Convert from pandas to numpy
vX = iris_df.to_numpy()
vY = iris_tf['species'].to_numpy()

### Presenting Dataset Samples

In [3]:
iris_df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,100.000000,100.000000,100.000000,100.000000
mean,5.471000,3.099000,2.861000,0.786000
std,0.641698,0.478739,1.449549,0.565153
min,4.300000,2.000000,1.000000,0.100000
25%,5.000000,2.800000,1.500000,0.200000
50%,5.400000,3.050000,2.450000,0.800000
75%,5.900000,3.400000,4.325000,1.300000
max,7.000000,4.400000,5.100000,1.800000


In [4]:
print(vY)

[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1]


### Splitting Data and Normalizing Input Vectors

Feature normalization (standardization to zero mean and unit variance) is applied to improve convergence of the subgradient updates.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(vX, vY, test_size=0.20, random_state=101)

# Standardize features (fit on train, apply to both)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

### Support Vector Machine

The separating hyperplane (Section 2.6) and the soft-margin SVM objective (Eq. 2.37) are:

$$H_0:\; \mathbf{w}^T \mathbf{x} + b = 0, \qquad
J = \frac{\lambda}{2} \lVert \mathbf{w} \rVert^2 + \sum_{i=1}^{N} \max\left\{0,\; 1-y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)}+b)\right\} $$

where $\lambda$ controls the trade-off between margin maximization and hinge-loss penalty (corresponding to $1/C$ in Eq. 2.37).

For samples inside the margin, i.e., $y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b) < 1$, the subgradients (Section 2.7) are:

$$ \frac{\partial J}{\partial \mathbf{w}} = \lambda \mathbf{w} - y^{(i)} \mathbf{x}^{(i)}, \qquad
\frac{\partial J}{\partial b} = -y^{(i)}$$

For samples outside the margin, only the regularization term contributes: $\frac{\partial J}{\partial \mathbf{w}} = \lambda \mathbf{w}$ (Eq. 2.38).

#### Define parameter container

In [6]:
class myParameters:
    def __init__(self):
        self.weight = np.zeros((4,))  # weight vector w for 4 input features
        self.bias = 0.0               # bias scalar b

#### Train model with subgradient descent

Unlike the per-sample stochastic updates in Eqs. (2.38–2.39), this implementation computes a **batch subgradient** by summing over all margin-violating samples per iteration.

In [ ]:
# Initialize parameters (w=0, b=0)
w = myParameters()

# Set hyperparameters
alpha = 0.001       # learning rate
r_lambda = 0.2      # regularization parameter lambda (= 1/C in Eq. 2.37)
n_epochs = 50       # number of training iterations

prev_sqr = 0.0

for epoch in range(n_epochs):
    
    ### START CODE HERE ###

    # Compute decision scores: w^T * x + b
    y_lin = None
    # Identify samples inside or on the margin: y*(w^T*x+b) < 1 (Eq. 2.36)
    in_margin = None

    # Compute subgradient for w: lambda*w - sum of y*x over margin samples (Eq. 2.39a)
    dJdw = None
    # Compute subgradient for b: -sum of y over margin samples (Eq. 2.39b)
    dJdb = None

    # Update parameters
    w.weight = None
    w.bias = None

    # Compute hinge loss for monitoring (Eq. 2.36)
    cost_J = None

    ### END CODE HERE ###

    # Decrease lambda when weight norm grows, for stable convergence
    sqr_wegt = np.sum(w.weight ** 2)
    if sqr_wegt > prev_sqr:
        r_lambda = r_lambda / 2
    prev_sqr = sqr_wegt

    if (epoch) % 10 == 0:
        print('Epoch: %5d,  cost: %11.8f' % (epoch, cost_J))

print('Epoch: %5d,  cost: %11.8f' % (epoch, cost_J))

Epoch:     0,  cost: 80.00000000
Epoch:    10,  cost:  1.93206256
Epoch:    20,  cost:  0.69400512
Epoch:    30,  cost:  0.34467206
Epoch:    40,  cost:  0.19346565
Epoch:    49,  cost:  0.12458708


In [8]:
res = [w.bias] + [w for w in w.weight]
exp = [0.1880000000, 0.2167578267, -0.5237369050, 0.4931223551, 0.5120213298]
if np.allclose(res,exp): print('test passed.')
else: print('test failed.')

test passed.


### Evaluate Model Performance

The predicted class label is determined by the sign of the decision function $\text{sign}(\mathbf{w}^T \mathbf{x} + b)$ (Eq. 2.28).

In [ ]:
def my_predict(w, X_test):

    ### START CODE HERE ###

    # Compute decision scores
    y_lin  = None
    # Apply sign function for class prediction (Eq. 2.28)
    y_pred = None

    ### END CODE HERE ###
    
    return y_pred

from sklearn.metrics import accuracy_score

y_pred = my_predict(w, X_test)

accuracy_score(y_pred, y_test)

1.0

In [10]:
if np.allclose(y_pred, y_test): print('test passed.')
else: print('test failed.')

test passed.


### Comparison with scikit-learn

scikit-learn's `SVC(kernel='linear')` solves the dual formulation (Eq. 2.35) using an optimized solver.

In [11]:
from sklearn.svm import SVC

svm = SVC(kernel='linear')

# Fit the model using the dual QP solver
svm.fit(X_train, y_train)

# Predict on the test set
s_pred = svm.predict(X_test)

accuracy_score(s_pred, y_test)

1.0

### Test Model with a Random Sample

Compare predictions from our subgradient implementation and scikit-learn's SVM on a single test sample.

In [12]:
idx = np.random.randint(X_test.shape[0])
test_in = np.expand_dims(X_test[idx], axis=0)

species = ['setosa', 'versicolor']

y_pred = my_predict(w, test_in)
s_pred = svm.predict(test_in)

# Convert labels from {-1, +1} to index {0, 1}
print('My prediction for Iris Species:', species[(y_pred[0]+1)//2])
print('SK prediction for Iris Species:', species[(s_pred[0]+1)//2])
print('Actual Iris Species:', species[(y_test[idx]+1)//2])

My prediction for Iris Species: versicolor
SK prediction for Iris Species: versicolor
Actual Iris Species: versicolor


(c) 2026 S. W. Lee